In [2]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx
#region Start
app = dash.Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP, #Wygląd
        "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.0/font/bootstrap-icons.css"
    ],
    suppress_callback_exceptions=True
)
DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['day_of_year'] = timestamp_pl.dt.day_of_year
df['weekday'] = timestamp_pl.dt.day_name('Pl')
df['weekday_number'] = timestamp_pl.dt.weekday
df['date'] = timestamp_pl.dt.date
df['localized_time_pl'] = df['timestamp_pl'].dt.tz_localize(None)

In [ ]:
dfAK = df[df.osoba == '']
dfAK.sample(3)

,timestamp,app,status,duration,category,osoba,device,timestamp_pl,day_of_year,weekday,weekday_number,date,localized_time_pl
116939,2026-01-02 14:01:49.344000+00:00,rstudio.exe,not-afk,36.595,studia,Kamil,PC,2026-01-02 15:01:49.344000+01:00,2,Piątek,4,2026-01-02,2026-01-02 15:01:49.344
33247,2026-01-08 11:14:23.268000+00:00,Ekran startowy One UI,not-afk,0.708,inne,Kamil,Phone,2026-01-08 12:14:23.268000+01:00,8,Czwartek,3,2026-01-08,2026-01-08 12:14:23.268
128200,2026-01-12 18:06:35.892000+00:00,Code.exe,not-afk,39.500,oglądanie filmów,Kamil,PC,2026-01-12 19:06:35.892000+01:00,12,Poniedziałek,0,2026-01-12,2026-01-12 19:06:35.892


Znajdźmy czasy pobudki z każdego dnia

In [4]:
poranki = dfAK[(dfAK['localized_time_pl'].dt.hour < 16) & (dfAK['localized_time_pl'].dt.hour > 5)]

In [5]:
pobudki = poranki.groupby('day_of_year').agg(End=('timestamp_pl','min'))
pobudki.sample(3)

,End
day_of_year,
347,2025-12-13 11:00:01.065000+01:00
351,2025-12-17 07:30:01.282000+01:00
358,2025-12-24 11:39:46.475000+01:00


Znajdźmy czasy pójście spać

In [6]:
wieczory = dfAK[(dfAK['localized_time_pl'].dt.hour > 16) | (dfAK['localized_time_pl'].dt.hour < 5)]
wieczory.sample(3)

,timestamp,app,status,duration,category,osoba,device,timestamp_pl,day_of_year,weekday,weekday_number,date,localized_time_pl
111155,2025-12-25 20:07:47.284000+00:00,brave.exe,not-afk,1.020,inne,Kamil,PC,2025-12-25 21:07:47.284000+01:00,359,Czwartek,3,2025-12-25,2025-12-25 21:07:47.284
104200,2025-12-15 20:51:34.059000+00:00,brave.exe,not-afk,5.066,studia,Kamil,PC,2025-12-15 21:51:34.059000+01:00,349,Poniedziałek,0,2025-12-15,2025-12-15 21:51:34.059
109520,2025-12-22 22:55:04.692000+00:00,brave.exe,not-afk,34.473,inne,Kamil,PC,2025-12-22 23:55:04.692000+01:00,356,Poniedziałek,0,2025-12-22,2025-12-22 23:55:04.692


Grupuje po dniach, przy czym godziny przed północy wliczam do następnego dnia

In [7]:
wieczory.loc[wieczory['localized_time_pl'].dt.hour>16,'day_of_year'] += 1
# Z pierszego stycznia staje się zero, więc trzeba dać na 365
wieczory.loc[wieczory['day_of_year'] == 366,'day_of_year'] = 1

In [8]:
zasypianie = wieczory.groupby('day_of_year').agg(Start=('timestamp_pl','max'))

In [9]:
zasypianie

,Start
day_of_year,
1,2026-01-01 02:37:20.772000+01:00
2,2026-01-02 00:56:22.714000+01:00
3,2026-01-03 01:14:59.574000+01:00
4,2026-01-04 00:40:27.655000+01:00
5,2026-01-05 03:01:49.109000+01:00
6,2026-01-06 00:32:17.113000+01:00
7,2026-01-07 00:00:00+01:00
8,2026-01-08 02:24:17.772000+01:00
9,2026-01-09 00:00:00+01:00


In [10]:
pobudki

,End
day_of_year,
1,2026-01-01 10:00:00.344000+01:00
2,2026-01-02 08:00:00.457000+01:00
3,2026-01-03 08:27:01.909000+01:00
4,2026-01-04 08:00:02.057000+01:00
5,2026-01-05 10:00:01.217000+01:00
6,2026-01-06 07:45:01.419000+01:00
7,2026-01-07 06:10:01.840000+01:00
8,2026-01-08 10:00:00.756000+01:00
9,2026-01-09 06:20:01.043000+01:00


In [11]:
SenAK = pd.concat([zasypianie,pobudki],axis=1).reset_index()

In [12]:
SenAK['Date'] = SenAK['End'].dt.strftime('%d.%m.%Y')
SenAK['StartTime'] = SenAK['Start'].dt.strftime('%H:%M')
SenAK['EndTime'] = SenAK['End'].dt.strftime('%H:%M')
SenAK.sample(3)

,day_of_year,Start,End,Date,StartTime,EndTime
25,346,2025-12-12 00:03:50.558000+01:00,2025-12-12 07:30:01.265000+01:00,12.12.2025,00:03,07:30
1,2,2026-01-02 00:56:22.714000+01:00,2026-01-02 08:00:00.457000+01:00,02.01.2026,00:56,08:00
36,357,2025-12-23 00:28:56.159000+01:00,2025-12-23 06:50:01.532000+01:00,23.12.2025,00:28,06:50


In [13]:
MojSen = SenAK[['Date','StartTime','EndTime']]
MojSen.head()

,Date,StartTime,EndTime
0,01.01.2026,02:37,10:00
1,02.01.2026,00:56,08:00
2,03.01.2026,01:14,08:27
3,04.01.2026,00:40,08:00
4,05.01.2026,03:01,10:00


In [14]:
#MojSen.to_csv('SenAK.csv', index=False)

In [28]:
MostFrequentApps = df.loc[df.category == 'inne',['app','duration']].groupby('app').sum().sort_values('duration',ascending=False) / 3600
MostFrequentApps.head(25)

,duration
app,
Chrome,147.722236
explorer.exe,61.187509
Firefox,47.746752
firefox.exe,28.787813
LockApp.exe,28.154515
brave.exe,27.141863
chrome.exe,15.854167
KR Battles,8.365367
aces.exe,6.656520
